In [ ]:
"""Responsibilities: register a frozen experiment module.
Implementation: compile source in an isolated module namespace.
Related Modules: later notebook cells import this registered module.
Declaration Index: None.
Variable Index: source is embedded Python; module is the registered namespace.
"""
import sys
import types
source = '"""职责：在 Kaggle GPU 上运行招聘双头模型的独立、固定协议可行性实验。\n实现：下载固定提交的数据；合并一致重复配对；按岗位隔离；比较文本与结构化基线，\n      冻结 Qwen 向量后训练双头网络，并评估资格方向的 top-10 精排。\n关联：不导入或改变 AiInterviewer 业务模块；kernel-metadata.json 负责私有远端执行。\n目录：\n- write_json：原子写入审计与结果。\n- bootstrap：安装指定 Transformers 版本，检查 CUDA，记录环境。\n- download_data：下载固定版本 CSV 并记录 SHA256。\n- consolidate_labels：验证并合并重复标签，保留各任务缺失状态。\n- split_jobs：生成固定、互斥的岗位划分。\n- parse_list：解析数据中的 Python 字符串列表。\n- profile_texts：生成无身份、无标签的模型输入。\n- structured_features：计算有明确语义的配对特征。\n- prepare_data：核对实体、标签、分布及跨划分不重叠。\n- ranking_metrics：在明确标注的候选池计算排名指标和查询覆盖。\n- evaluate_scores：计算每任务 AP、AUC 与排名指标。\n- lexical_baseline：仅用训练侧语料拟合 TF-IDF。\n- encode_profiles：按固定上限编码并记录截断比例。\n- build_pair_features：组合冻结向量与结构化输入。\n- DualHeadModel：共享表示和偏好/资格两个预测头。\n- DualHeadModel.__init__：创建固定网络层。\n- DualHeadModel.forward：产生两个未校准 logits。\n- train_dual_head：以标签掩码训练并仅按验证 AP 选检查点。\n- structured_baseline：拟合训练侧标准化与两个逻辑回归。\n- rerank_qualification：对测试岗位语义召回的 top-10 做精排。\n- write_report：保存可读指标表、边界及配置。\n- main：按依赖顺序执行实验，失败时记录阶段和堆栈并重新抛出。\n变量索引：\n- CONFIG：冻结的数据提交、模型修订、划分、预处理与训练配置。\n- FILES：允许下载的七个公开合成数据文件。\n- STRUCTURED_NAMES：结构化特征的顺序与含义。\n- LOGGER：仅记录阶段、尺寸、统计与错误，不记录凭据和档案正文。\n约束：要求 CUDA；不自动换模型、减小输入或重试失败步骤。结果属于合成数据上的\n      新岗位实验；不复现作者原始划分，不把未观测标签当负例。\n"""\n\nfrom __future__ import annotations\n\nimport ast\nimport copy\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport logging\nimport os\nimport platform\nimport random\nimport subprocess\nimport sys\nimport time\nimport urllib.request\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score, roc_auc_score\nfrom sklearn.preprocessing import StandardScaler\n\nCONFIG = {\n    "protocol": "jobrec-cold-job-feasibility-v1",\n    "seed": 42,\n    "data_commit": "c742feea3730e8d34ec2e8ae7e58c8c0ee8a53fd",\n    "embedding_model": "Qwen/Qwen3-Embedding-0.6B",\n    "embedding_revision": "97b0c614be4d77ee51c0cef4e5f07c00f9eb65b3",\n    "reranker_model": "Qwen/Qwen3-Reranker-0.6B",\n    "reranker_revision": "e61197ed45024b0ed8a2d74b80b4d909f1255473",\n    "transformers_version": "4.56.2",\n    "job_split_counts": [70, 15, 15],\n    "embedding_max_length": 384,\n    "embedding_batch_size": 8,\n    "embedding_dtype": "float16",\n    "attention": "sdpa",\n    "job_instruction": "Given a job description, retrieve candidates qualified for the job.",\n    "epochs": 12,\n    "training_batch_size": 512,\n    "learning_rate": 0.001,\n    "weight_decay": 0.0001,\n    "hidden_sizes": [128, 32],\n    "dropout": 0.1,\n    "tfidf_max_features": 20000,\n    "tfidf_min_df": 2,\n    "logistic_max_iter": 2000,\n    "rerank_top_k": 10,\n    "rerank_batch_size": 4,\n    "rerank_query_tokens": 160,\n    "rerank_document_tokens": 320,\n    "rerank_max_length": 768,\n    "ranking_k": [5, 10],\n    "selection_metric": "mean_validation_average_precision_over_two_observed_tasks",\n}\nFILES = (\n    "Final_users.csv", "Final_items.csv", "train.csv", "val.csv", "test.csv",\n    "qual_train.csv", "qual_test.csv",\n)\nSTRUCTURED_NAMES = (\n    "skill_coverage", "interest_matches_industry", "major_accepted", "work_mode_matches",\n    "gpa_margin", "experience_months_margin", "academic_level_margin",\n    "weekly_hours_margin", "commitment_months_margin", "publications_margin",\n    "summer_commitment",\n)\nLOGGER = logging.getLogger("jobrec")\n\n\ndef write_json(path: Path, value: object) -> None:\n    """功能：原子保存 JSON；输入 path/value；输出无，写文件。\n    逻辑：同目录临时文件后 replace；约束：拒绝非有限数，写入失败向调用方传播。\n    """\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")\n    temporary.replace(path)\n\n\ndef bootstrap(output: Path) -> None:\n    """功能：准备指定依赖并检查 GPU；输入输出目录；输出无。\n    逻辑：显式安装固定 Transformers、设种子并记录平台包版本；约束：无 GPU 即失败，\n    不更换硬件或参数。Kaggle 预装 torch/numpy 等保留并记录，不能视为完整环境锁定。\n    """\n    os.environ["HF_HOME"] = "/tmp/jobrec_hf"\n    os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"\n    os.environ["HF_HUB_DISABLE_XET"] = "1"\n    os.environ["TOKENIZERS_PARALLELISM"] = "false"\n    subprocess.run([\n        sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",\n        "transformers==" + CONFIG["transformers_version"],\n    ], check=True)\n    if not torch.cuda.is_available():\n        raise RuntimeError("Protocol requires CUDA; refusing a CPU substitution.")\n    random.seed(CONFIG["seed"])\n    np.random.seed(CONFIG["seed"])\n    torch.manual_seed(CONFIG["seed"])\n    torch.cuda.manual_seed_all(CONFIG["seed"])\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n    torch.set_num_threads(2)\n    environment = {\n        "python": sys.version, "platform": platform.platform(),\n        "packages": {name: importlib.metadata.version(name) for name in (\n            "torch", "numpy", "pandas", "scikit-learn", "transformers", "tokenizers",\n            "huggingface-hub", "safetensors",\n        )},\n        "gpu_names": [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],\n        "used_device": "cuda:0", "cuda": torch.version.cuda,\n        "note": "Seeds fixed; bitwise reproducibility across GPU/library versions is not guaranteed.",\n    }\n    write_json(output / "environment.json", environment)\n    LOGGER.info("Environment ready: GPUs=%s", environment["gpu_names"])\n\n\ndef download_data(output: Path) -> dict[str, pd.DataFrame]:\n    """功能：取得固定提交的七份 CSV；输入输出目录；输出文件名到 DataFrame。\n    逻辑：请求公开 URL，逐文件计算哈希与行数；约束：无重试、不写凭据和原始档案副本。\n    """\n    import io\n\n    frames, manifest = {}, {}\n    base = "https://raw.githubusercontent.com/brycekan123/DualOptimization_jobrec/"\n    for filename in FILES:\n        url = base + CONFIG["data_commit"] + "/dataset/" + filename\n        with urllib.request.urlopen(url, timeout=60) as response:\n            raw = response.read()\n        frame = pd.read_csv(io.BytesIO(raw))\n        frames[filename] = frame\n        manifest[filename] = {"url": url, "sha256": hashlib.sha256(raw).hexdigest(), "rows": len(frame)}\n        LOGGER.info("Downloaded %s: %d rows", filename, len(frame))\n    write_json(output / "data_manifest.json", manifest)\n    return frames\n\n\ndef consolidate_labels(frames: dict[str, pd.DataFrame]) -> tuple[pd.DataFrame, dict]:\n    """功能：构造唯一配对表及原始重复审计；输入 CSV 表；输出合并表/审计。\n    逻辑：相同任务同一配对只保留一致标签；不同任务 outer join 保留 NaN 掩码。\n    约束：冲突或非二元标签直接报错；不使用源 score、不把缺失资格补成 0。\n    """\n    tables, audit = [], {}\n    for task, names in (("pref", ("train.csv", "val.csv", "test.csv")),\n                        ("qual", ("qual_train.csv", "qual_test.csv"))):\n        label = "label_" + task\n        combined = pd.concat([frames[name][["user", "item", label]] for name in names], ignore_index=True)\n        if combined.isna().any().any() or not set(combined[label].unique()).issubset({0, 1}):\n            raise ValueError(f"Invalid identifiers or binary labels for {task}")\n        if (combined.groupby(["user", "item"])[label].nunique() > 1).any():\n            raise ValueError(f"Conflicting duplicate labels for {task}")\n        unique = combined.drop_duplicates(["user", "item"]).copy()\n        audit[task] = {"source_rows": len(combined), "unique_pairs": len(unique),\n                       "consistent_duplicates_collapsed": len(combined) - len(unique)}\n        tables.append(unique)\n    merged = tables[0].merge(tables[1], on=["user", "item"], how="outer", validate="one_to_one")\n    return merged.sort_values(["user", "item"]).reset_index(drop=True), audit\n\n\ndef split_jobs(job_ids: list[str]) -> dict[str, str]:\n    """功能：固定冷岗位划分；输入岗位 ID；输出岗位到 split 映射。\n    逻辑：排序后使用独立 seed 置换；约束：要求恰好100个不同岗位，比例固定70/15/15。\n    """\n    ids = sorted(job_ids)\n    if len(ids) != 100 or len(set(ids)) != 100:\n        raise ValueError("Expected exactly 100 distinct jobs.")\n    shuffled = np.random.default_rng(CONFIG["seed"]).permutation(ids)\n    return {str(job): ("train" if i < 70 else "validation" if i < 85 else "test")\n            for i, job in enumerate(shuffled)}\n\n\ndef parse_list(value: str) -> list[str]:\n    """功能：解析源数据列表；输入 Python 列表文本；输出字符串列表。\n    逻辑：literal_eval 后检查类型；约束：格式错误直接抛出，不隐式猜测分隔符。\n    """\n    parsed = ast.literal_eval(value)\n    if not isinstance(parsed, list) or not all(isinstance(item, str) for item in parsed):\n        raise ValueError("Expected a list of strings in profile.")\n    return parsed\n\n\ndef profile_texts(users: pd.DataFrame, jobs: pd.DataFrame) -> tuple[list[str], list[str]]:\n    """功能：形成文本输入；输入完整档案表；输出候选文本/岗位查询文本。\n    逻辑：技能/兴趣/教育/经验在前、合成描述在后；岗位使用固定资格检索指令。\n    约束：不输入 ID、匹配标签、源 score 或虚构面试证据；后续按 token 上限截断。\n    """\n    candidate_texts = [\n        f"Skills: {\', \'.join(parse_list(row.skills))}. Interests: {\', \'.join(parse_list(row.interests))}. "\n        f"Education: {row.academic_level}, {row.masters_major}, {row.undergrad_major}. "\n        f"Experience months: {row.months_experience}. Profile: {row.student_llm_output}"\n        for row in users.itertuples()\n    ]\n    job_texts = [\n        f"Instruct: {CONFIG[\'job_instruction\']}\\nQuery: "\n        f"Industry: {row.industry}. Skills: {\', \'.join(parse_list(row.required_skills))}. "\n        f"Education: {row.min_academic_level}. Experience months: {row.min_months_experience}. "\n        f"Description: {row.job_llm_output}"\n        for row in jobs.itertuples()\n    ]\n    return candidate_texts, job_texts\n\n\ndef structured_features(pairs: pd.DataFrame, users: pd.DataFrame, jobs: pd.DataFrame) -> np.ndarray:\n    """功能：计算11项显式匹配特征；输入配对和档案；输出 float32 矩阵。\n    逻辑：技能覆盖、偏好/专业/模式匹配及数值余量；约束：资格标签可能由类似规则生成，\n    因而本基线仅度量合成任务可学习性。没有人口属性和不存在的面试分。\n    """\n    user_map = users.set_index("student_id").to_dict("index")\n    job_map = jobs.set_index("job_id").to_dict("index")\n    levels = {"UG1": 1, "UG2": 2, "UG3": 3, "UG4": 4, "MS1": 5, "MS2": 6,\n              "PhD1": 7, "PhD2": 8, "PhD3": 9, "PhD4": 10, "PhD5": 11}\n    result = []\n    for row in pairs.itertuples():\n        user, job = user_map[row.user], job_map[row.item]\n        skills, required = set(parse_list(user["skills"])), set(parse_list(job["required_skills"]))\n        if not required:\n            raise ValueError("Empty required skill list is outside the frozen protocol.")\n        majors = {user["masters_major"], user["second_major"], user["undergrad_major"]}\n        features = [len(skills & required) / len(required),\n                    float(job["industry"] in parse_list(user["interests"])),\n                    float(bool(majors & set(parse_list(job["acceptable_majors"])))),\n                    float(user["in_person_commitment"] == job["job_in_person_commitment"]),\n                    float(user["gpa"]) - float(job["min_gpa"]),\n                    float(user["months_experience"]) - float(job["min_months_experience"]),\n                    float(levels[user["academic_level"]] - levels[job["min_academic_level"]]),\n                    float(user["hours_per_week"]) - float(job["min_hours_per_week"]),\n                    float(user["length_of_commitment"]) - float(job["min_length_of_commitment"]),\n                    float(user["num_publications"]) - float(job["min_num_publications"]),\n                    float(str(user["commit_to_summer"]).lower() == "true")]\n        result.append(features)\n    matrix = np.asarray(result, dtype=np.float32)\n    if not np.isfinite(matrix).all():\n        raise ValueError("Non-finite structured features.")\n    return matrix\n\n\ndef prepare_data(frames: dict[str, pd.DataFrame], output: Path) -> tuple:\n    """功能：完成实体映射与划分审计；输入源表/输出路径；输出全部训练准备对象。\n    逻辑：合并配对、岗位隔离、验证每任务每划分均有正负例；约束：候选人允许跨划分，\n    所以这是冷岗位而非冷候选人实验；候选池仅包含源数据明确标注的配对。\n    """\n    users = frames["Final_users.csv"].sort_values("student_id").reset_index(drop=True)\n    jobs = frames["Final_items.csv"].sort_values("job_id").reset_index(drop=True)\n    if users.student_id.duplicated().any() or jobs.job_id.duplicated().any():\n        raise ValueError("Entity IDs must be unique.")\n    pairs, audit = consolidate_labels(frames)\n    split_map = split_jobs(jobs.job_id.tolist())\n    pairs["split"] = pairs.item.map(split_map)\n    pairs["u_idx"] = pairs.user.map(dict(zip(users.student_id, users.index)))\n    pairs["j_idx"] = pairs.item.map(dict(zip(jobs.job_id, jobs.index)))\n    if pairs[["split", "u_idx", "j_idx"]].isna().any().any():\n        raise ValueError("Unknown entity in label table.")\n    for split in ("train", "validation", "test"):\n        audit[split] = {}\n        for task in ("pref", "qual"):\n            observed = pairs.loc[pairs.split.eq(split), "label_" + task].dropna()\n            if set(observed.unique()) != {0, 1}:\n                raise ValueError(f"Both classes required: {split}/{task}")\n            audit[split][task] = {"pairs": len(observed), "positives": int(observed.sum())}\n    audit["protocol_note"] = "Disjoint jobs; shared candidates; observed labeled candidate pools only."\n    audit["structured_feature_names"] = STRUCTURED_NAMES\n    write_json(output / "split_manifest.json", split_map)\n    write_json(output / "data_audit.json", audit)\n    pairs.to_csv(output / "pair_splits.csv", index=False)\n    texts = profile_texts(users, jobs)\n    features = structured_features(pairs, users, jobs)\n    LOGGER.info("Data audit: %s", json.dumps(audit))\n    return pairs, users, jobs, texts, features\n\n\ndef ranking_metrics(frame: pd.DataFrame, scores: np.ndarray, task: str) -> dict:\n    """功能：计算明确候选池的宏平均排名；输入已观察标签表/分数/task；输出指标字典。\n    逻辑：pref 按候选人分组，qual 按岗位分组；无正例组不计均值并单独报告；同分按稳定\n    输入顺序打破。约束：不是全库检索；nDCG/Recall 的分母只来自已标注候选池。\n    """\n    scored = frame.assign(pred=np.asarray(scores))\n    group_column = "user" if task == "pref" else "item"\n    label_column = "label_" + task\n    per_query, pool_sizes, all_negative = [], [], 0\n    for _, group in scored.groupby(group_column, sort=True):\n        labels = group[label_column].to_numpy()\n        positives = int(labels.sum())\n        if positives == 0:\n            all_negative += 1\n            continue\n        ranked = group.sort_values("pred", ascending=False, kind="stable")[label_column].to_numpy()\n        values = {}\n        for k in CONFIG["ranking_k"]:\n            limit = min(k, len(ranked))\n            discount = 1 / np.log2(np.arange(limit) + 2)\n            ideal = (1 / np.log2(np.arange(min(positives, limit)) + 2)).sum()\n            values[f"ndcg@{k}"] = float((ranked[:limit] * discount).sum() / ideal)\n            values[f"recall@{k}"] = float(ranked[:limit].sum() / positives)\n        per_query.append(values)\n        pool_sizes.append(len(group))\n    if not per_query:\n        raise ValueError(f"No evaluable positive queries for {task}")\n    return {**{name: float(np.mean([row[name] for row in per_query])) for name in per_query[0]},\n            "positive_queries": len(per_query), "all_negative_queries_excluded": all_negative,\n            "median_labeled_pool": float(np.median(pool_sizes))}\n\n\ndef evaluate_scores(pairs: pd.DataFrame, scores: np.ndarray, split: str) -> dict:\n    """功能：按任务和数据划分评测；输入配对表、N×2分数、split；输出两任务指标。\n    逻辑：只选显式标签；约束：sigmoid/排序分数不被当作已校准业务概率。\n    """\n    result = {}\n    for column, task in enumerate(("pref", "qual")):\n        selected = pairs.split.eq(split) & pairs["label_" + task].notna()\n        frame = pairs.loc[selected]\n        labels = frame["label_" + task].to_numpy()\n        values = scores[selected.to_numpy(), column]\n        result[task] = {"average_precision": float(average_precision_score(labels, values)),\n                        "roc_auc": float(roc_auc_score(labels, values)),\n                        "positive_prevalence": float(labels.mean()), "pairs": len(frame),\n                        **ranking_metrics(frame, values, task)}\n    return result\n\n\ndef lexical_baseline(pairs: pd.DataFrame, texts: tuple, output: Path) -> np.ndarray:\n    """功能：计算 TF-IDF 配对余弦；输入配对/文本/目录；输出N×2共享语义分数。\n    逻辑：只在训练配对涉及的实体文本上拟合词表，验证测试仅 transform；约束：此分数\n    没有学习双边意愿。记录词表规模，不序列化含档案词汇的模型。\n    """\n    candidate_texts, job_texts = texts\n    train = pairs.loc[pairs.split.eq("train")]\n    corpus = [candidate_texts[i] for i in sorted(train.u_idx.unique())]\n    corpus += [job_texts[i] for i in sorted(train.j_idx.unique())]\n    vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=CONFIG["tfidf_min_df"],\n                                 max_features=CONFIG["tfidf_max_features"], dtype=np.float32)\n    vectorizer.fit(corpus)\n    u, j = vectorizer.transform(candidate_texts), vectorizer.transform(job_texts)\n    values = np.asarray(u[pairs.u_idx].multiply(j[pairs.j_idx]).sum(axis=1)).ravel()\n    write_json(output / "lexical_audit.json", {"vocabulary_size": len(vectorizer.vocabulary_),\n               "training_entities_only": True})\n    return np.column_stack([values, values])\n\n\ndef encode_profiles(texts: tuple, output: Path) -> tuple[np.ndarray, np.ndarray]:\n    """功能：生成冻结 Qwen 表示；输入候选/岗位文本与目录；输出两侧归一化向量。\n    逻辑：固定修订、左 padding、最后有效 token 池化；FP16/SDPA/cuda:0。\n    约束：固定384 tokens，不因内存错误调整；仅保存向量和截断统计，不输出档案正文。\n    """\n    from transformers import AutoModel, AutoTokenizer\n\n    tokenizer = AutoTokenizer.from_pretrained(CONFIG["embedding_model"],\n                 revision=CONFIG["embedding_revision"], padding_side="left")\n    model = AutoModel.from_pretrained(CONFIG["embedding_model"],\n             revision=CONFIG["embedding_revision"], torch_dtype=torch.float16,\n             attn_implementation=CONFIG["attention"]).to("cuda:0").eval()\n    all_texts = texts[0] + texts[1]\n    chunks, lengths, started = [], [], time.monotonic()\n    batch_size = CONFIG["embedding_batch_size"]\n    for offset in range(0, len(all_texts), batch_size):\n        batch = all_texts[offset:offset + batch_size]\n        raw_tokens = tokenizer(batch, truncation=False, add_special_tokens=True)["input_ids"]\n        lengths.extend(len(row) for row in raw_tokens)\n        inputs = tokenizer(batch, padding=True, truncation=True,\n                  max_length=CONFIG["embedding_max_length"], return_tensors="pt").to("cuda:0")\n        with torch.inference_mode():\n            hidden = model(**inputs).last_hidden_state[:, -1]\n            encoded = torch.nn.functional.normalize(hidden.float(), dim=-1)\n        chunks.append(encoded.cpu().numpy())\n        if offset % (batch_size * 25) == 0:\n            LOGGER.info("Embedding %d/%d elapsed=%.1fs", offset + len(batch), len(all_texts), time.monotonic() - started)\n    embeddings = np.concatenate(chunks)\n    if not np.isfinite(embeddings).all():\n        raise ValueError("Non-finite embeddings")\n    audit = {"entities": len(all_texts), "dimension": embeddings.shape[1],\n             "elapsed_seconds": time.monotonic() - started,\n             "truncated_entities": int(np.sum(np.asarray(lengths) > CONFIG["embedding_max_length"])),\n             "median_original_tokens": float(np.median(lengths)),\n             "semantic_baseline": "job-query instruction; same qualification-oriented cosine in both directions"}\n    write_json(output / "embedding_audit.json", audit)\n    np.savez_compressed(output / "embeddings.npz", candidates=embeddings[:len(texts[0])], jobs=embeddings[len(texts[0]):])\n    del model, inputs, hidden, encoded\n    gc.collect()\n    torch.cuda.empty_cache()\n    LOGGER.info("Embedding completed: %s", audit)\n    return embeddings[:len(texts[0])], embeddings[len(texts[0]):]\n\n\ndef build_pair_features(pairs: pd.DataFrame, embeddings: tuple, structured: np.ndarray,\n                        output: Path) -> tuple[np.ndarray, np.ndarray]:\n    """功能：合并冻结向量和结构化特征；输入配对/向量/特征/目录；输出网络输入/余弦。\n    逻辑：abs(u-v)、逐维乘积、训练侧标准化的结构化特征；约束：标准化不能使用验证测试。\n    """\n    u, j = embeddings[0][pairs.u_idx], embeddings[1][pairs.j_idx]\n    scaler = StandardScaler().fit(structured[pairs.split.eq("train")])\n    features = np.concatenate([np.abs(u - j), u * j, scaler.transform(structured)], axis=1).astype(np.float32)\n    cosine = (u * j).sum(axis=1)\n    write_json(output / "feature_scaler.json", {"names": STRUCTURED_NAMES,\n               "mean": scaler.mean_.tolist(), "scale": scaler.scale_.tolist()})\n    return features, np.column_stack([cosine, cosine])\n\n\nclass DualHeadModel(torch.nn.Module):\n    """功能：偏好与资格共享表示网络；逻辑：MLP trunk 后分别线性输出。\n    约束：两头独立标签掩码；实例状态 trunk/heads 只含训练权重，不含实体 ID。\n    """\n\n    def __init__(self, feature_size: int):\n        """功能：初始化网络；输入特征维度；输出新实例。\n        逻辑：固定128/32隐藏层和dropout；约束：网络参数与 CONFIG 一致。\n        """\n        super().__init__()\n        first, second = CONFIG["hidden_sizes"]\n        self.trunk = torch.nn.Sequential(torch.nn.Linear(feature_size, first), torch.nn.ReLU(),\n                     torch.nn.Dropout(CONFIG["dropout"]), torch.nn.Linear(first, second), torch.nn.ReLU())\n        self.heads = torch.nn.ModuleList([torch.nn.Linear(second, 1), torch.nn.Linear(second, 1)])\n\n    def forward(self, features: torch.Tensor) -> torch.Tensor:\n        """功能：输出偏好/资格 logits；输入 B×D；输出 B×2。\n        逻辑：共享 trunk 后拼接两个线性头；约束：推理调用方负责 eval 和 sigmoid。\n        """\n        hidden = self.trunk(features)\n        return torch.cat([head(hidden) for head in self.heads], dim=1)\n\n\ndef train_dual_head(pairs: pd.DataFrame, features: np.ndarray, output: Path) -> np.ndarray:\n    """功能：训练双头模型；输入配对/特征/目录；输出所有配对两头分数。\n    逻辑：缺失标签通过 mask 排除，训练侧正类权重，12轮中按验证双任务平均AP选最佳；\n    约束：不查测试选择模型，不早停或自动调整参数；测试结果只在训练结束后计算。\n    """\n    model = DualHeadModel(features.shape[1]).to("cuda:0")\n    x = torch.from_numpy(features).to("cuda:0")\n    targets = pairs[["label_pref", "label_qual"]].to_numpy(dtype=np.float32)\n    observed = np.isfinite(targets)\n    y = torch.from_numpy(np.nan_to_num(targets, nan=0)).to("cuda:0")\n    mask = torch.from_numpy(observed).to("cuda:0")\n    train_ids = np.flatnonzero(pairs.split.eq("train"))\n    val_ids = np.flatnonzero(pairs.split.eq("validation"))\n    weights = []\n    for column in range(2):\n        labels = targets[train_ids, column]\n        labels = labels[np.isfinite(labels)]\n        weights.append(float((labels == 0).sum() / (labels == 1).sum()))\n    criterion = torch.nn.BCEWithLogitsLoss(reduction="none", pos_weight=torch.tensor(weights, device="cuda:0"))\n    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])\n    best_score, best_state, history = -1.0, None, []\n    rng = np.random.default_rng(CONFIG["seed"])\n    for epoch in range(1, CONFIG["epochs"] + 1):\n        model.train()\n        losses, order = [], rng.permutation(train_ids)\n        for start in range(0, len(order), CONFIG["training_batch_size"]):\n            ids = order[start:start + CONFIG["training_batch_size"]]\n            optimizer.zero_grad(set_to_none=True)\n            element_loss = criterion(model(x[ids]), y[ids])\n            task_losses = [(element_loss[:, c] * mask[ids, c]).sum() / mask[ids, c].sum()\n                           for c in range(2) if mask[ids, c].any()]\n            loss = torch.stack(task_losses).mean()\n            loss.backward()\n            optimizer.step()\n            losses.append(float(loss.item()))\n        model.eval()\n        with torch.inference_mode():\n            val_scores = torch.sigmoid(model(x[val_ids])).cpu().numpy()\n        aps = [float(average_precision_score(targets[val_ids, c][observed[val_ids, c]],\n               val_scores[:, c][observed[val_ids, c]])) for c in range(2)]\n        validation_mean = float(np.mean(aps))\n        row = {"epoch": epoch, "mean_batch_loss": float(np.mean(losses)), "validation_ap": aps}\n        history.append(row)\n        LOGGER.info("Training epoch: %s", row)\n        if validation_mean > best_score:\n            best_score = validation_mean\n            best_state = copy.deepcopy(model.state_dict())\n            selected_epoch = epoch\n    if best_state is None:\n        raise RuntimeError("No finite validation checkpoint")\n    model.load_state_dict(best_state)\n    torch.save({"state_dict": {k: v.cpu() for k, v in best_state.items()},\n                "feature_size": features.shape[1], "config": CONFIG}, output / "dual_head.pt")\n    with torch.inference_mode():\n        scores = torch.sigmoid(model(x)).cpu().numpy()\n    write_json(output / "training_history.json", {"epochs": history, "selected_epoch": selected_epoch,\n               "selection_validation_mean_ap": best_score, "positive_weights": weights})\n    del model, x, y, mask, optimizer, best_state\n    gc.collect()\n    torch.cuda.empty_cache()\n    return scores\n\n\ndef structured_baseline(pairs: pd.DataFrame, features: np.ndarray, output: Path) -> np.ndarray:\n    """功能：结构化逻辑回归基线；输入配对/原始11维/目录；输出N×2分数。\n    逻辑：各头只在各自训练标签上拟合标准化和逻辑回归；约束：无任务标签传播。\n    """\n    predictions, records = [], {}\n    for task in ("pref", "qual"):\n        observed = pairs.split.eq("train") & pairs["label_" + task].notna()\n        scaler = StandardScaler().fit(features[observed])\n        x = scaler.transform(features)\n        model = LogisticRegression(class_weight="balanced", random_state=CONFIG["seed"],\n                                    max_iter=CONFIG["logistic_max_iter"])\n        model.fit(x[observed], pairs.loc[observed, "label_" + task])\n        if int(model.n_iter_[0]) >= CONFIG["logistic_max_iter"]:\n            raise RuntimeError(f"Logistic regression did not converge: {task}")\n        predictions.append(model.predict_proba(x)[:, 1])\n        records[task] = {"names": STRUCTURED_NAMES, "coef": model.coef_[0].tolist(),\n                         "intercept": model.intercept_.tolist(), "iterations": model.n_iter_.tolist()}\n    write_json(output / "structured_logistic.json", records)\n    return np.column_stack(predictions)\n\n\ndef rerank_qualification(pairs: pd.DataFrame, texts: tuple, cosine: np.ndarray, output: Path) -> dict:\n    """功能：资格方向两阶段精排；输入配对/文本/余弦/目录；输出固定测试候选池的指标。\n    逻辑：各岗位语义 top10，Qwen yes/no logits 重排前缀，尾部保持原次序。\n    约束：固定 query/doc 各160/320token；不是全库召回；不报告伪概率AUC，不调整候选池。\n    """\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    frame = pairs.loc[pairs.split.eq("test") & pairs.label_qual.notna()].copy()\n    frame["semantic"] = cosine[frame.index, 1]\n    tokenizer = AutoTokenizer.from_pretrained(CONFIG["reranker_model"],\n                 revision=CONFIG["reranker_revision"], padding_side="left")\n    model = AutoModelForCausalLM.from_pretrained(CONFIG["reranker_model"],\n             revision=CONFIG["reranker_revision"], torch_dtype=torch.float16,\n             attn_implementation=CONFIG["attention"]).to("cuda:0").eval()\n    prefix = (\'<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query \'\n              \'and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\\n<|im_start|>user\\n\')\n    suffix = "<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n"\n    prefix_ids, suffix_ids = tokenizer.encode(prefix, add_special_tokens=False), tokenizer.encode(suffix, add_special_tokens=False)\n    selected, payloads = [], []\n    ranked_groups = []\n    for _, group in frame.groupby("item", sort=True):\n        ranked = group.sort_values("semantic", ascending=False, kind="stable")\n        ranked_groups.append(ranked.index.tolist())\n        for row in ranked.head(CONFIG["rerank_top_k"]).itertuples():\n            query_ids = tokenizer.encode(texts[1][row.j_idx], add_special_tokens=False)[:CONFIG["rerank_query_tokens"]]\n            doc_ids = tokenizer.encode(texts[0][row.u_idx], add_special_tokens=False)[:CONFIG["rerank_document_tokens"]]\n            body = f"<Instruct>: {CONFIG[\'job_instruction\']}\\n<Query>: {tokenizer.decode(query_ids)}\\n<Document>: {tokenizer.decode(doc_ids)}"\n            ids = prefix_ids + tokenizer.encode(body, add_special_tokens=False) + suffix_ids\n            if len(ids) > CONFIG["rerank_max_length"]:\n                raise ValueError("Reranker formatted sequence exceeds frozen limit")\n            selected.append(row.Index)\n            payloads.append(ids)\n    values, started = [], time.monotonic()\n    yes, no = tokenizer.convert_tokens_to_ids("yes"), tokenizer.convert_tokens_to_ids("no")\n    for start in range(0, len(payloads), CONFIG["rerank_batch_size"]):\n        encoded = tokenizer.pad({"input_ids": payloads[start:start + CONFIG["rerank_batch_size"]]},\n                                padding=True, return_tensors="pt").to("cuda:0")\n        with torch.inference_mode():\n            logits = model(**encoded, logits_to_keep=1).logits[:, -1, :]\n            scores = torch.softmax(logits[:, [no, yes]].float(), dim=-1)[:, 1]\n        values.extend(scores.cpu().tolist())\n        if start % 40 == 0:\n            LOGGER.info("Reranking %d/%d", start + len(scores), len(payloads))\n    by_index = dict(zip(selected, values))\n    final = pd.Series(index=frame.index, dtype=float)\n    for indexes in ranked_groups:\n        head = indexes[:CONFIG["rerank_top_k"]]\n        reordered = sorted(head, key=by_index.__getitem__, reverse=True) + indexes[CONFIG["rerank_top_k"]:]\n        for rank, index in enumerate(reordered):\n            final.loc[index] = len(reordered) - rank\n    result = {"qual": ranking_metrics(frame, final.loc[frame.index].to_numpy(), "qual"),\n              "reranked_pairs": len(selected), "elapsed_seconds": time.monotonic() - started,\n              "scope": "test qualification labeled pools; semantic top10 prefix only"}\n    write_json(output / "reranker_audit.json", result)\n    pd.DataFrame({"pair_index": selected, "reranker_yes_score": values}).to_csv(output / "reranker_scores.csv", index=False)\n    del model\n    gc.collect()\n    torch.cuda.empty_cache()\n    return result\n\n\ndef write_report(results: dict, output: Path) -> None:\n    """功能：输出可读报告；输入指标/目录；输出无，保存 Markdown。\n    逻辑：分别展示任务及候选池指标；约束：不把合成标签结果写成真实招聘或面试增益。\n    """\n    lines = ["# JobRec cold-job feasibility v1", "", "Completed on Kaggle; frozen Qwen embeddings + masked dual-head MLP.",\n             "", "| Model | Task | AP | AUC | NDCG@5 | Recall@5 | Queries | Median labeled pool |",\n             "|---|---|---:|---:|---:|---:|---:|---:|"]\n    for name, metrics in results["models"].items():\n        for task, row in metrics.items():\n            lines.append(f"| {name} | {task} | {row[\'average_precision\']:.4f} | {row[\'roc_auc\']:.4f} | "\n                         f"{row[\'ndcg@5\']:.4f} | {row[\'recall@5\']:.4f} | {row[\'positive_queries\']} | {row[\'median_labeled_pool\']:.1f} |")\n    lines += ["", "## Reranker", "", json.dumps(results["reranker"], indent=2), "", "## Limits", "",\n              "- Fully synthetic, rule-derived supervision; no real hiring outcome validation.",\n              "- Jobs split 70/15/15; candidates may recur. This is a new cold-job protocol, not paper reproduction.",\n              "- Source repeated labels collapsed only after consistency checks; unknown labels stay masked.",\n              "- Rankings use observed labeled pools, not the full catalog; both task pools differ.",\n              "- Qualification labels are conditional on the source selection process and are very sparse.",\n              "- No interview features exist in this dataset; interview benefit has not been tested.",\n              "- Text length, seed, checkpoints and training settings were fixed before execution.",\n              "- AP prevalence baselines and evaluated query counts are in metrics.json.",\n              "- Dataset repository license was not explicit; kernel remains private."]\n    (output / "RESULTS.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n\n\ndef main() -> None:\n    """功能：执行完整实验；输入 Kaggle GPU/网络环境；输出 /kaggle/working/jobrec_v1。\n    逻辑：各阶段及时保存审计，最终保存模型/预测/指标；约束：任何异常记录当前阶段并\n    重新抛出，不自动重跑、降级或改参数。故障输出保留已完成阶段的证据。\n    """\n    output = Path("/kaggle/working/jobrec_v1")\n    output.mkdir(parents=True, exist_ok=True)\n    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",\n                        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(output / "execution.log", encoding="utf-8")])\n    stage, started = "bootstrap", time.monotonic()\n    write_json(output / "config.json", CONFIG)\n    try:\n        bootstrap(output)\n        stage = "download_and_audit"\n        frames = download_data(output)\n        pairs, users, jobs, texts, structured = prepare_data(frames, output)\n        stage = "baselines"\n        lexical = lexical_baseline(pairs, texts, output)\n        logistic = structured_baseline(pairs, structured, output)\n        stage = "frozen_qwen_embedding"\n        embeddings = encode_profiles(texts, output)\n        features, cosine = build_pair_features(pairs, embeddings, structured, output)\n        stage = "dual_head_training"\n        neural = train_dual_head(pairs, features, output)\n        predictions = {"tfidf": lexical, "structured_logistic": logistic,\n                       "qwen_embedding": cosine, "dual_head_mlp": neural}\n        results = {"protocol": CONFIG["protocol"],\n                   "models": {name: evaluate_scores(pairs, score, "test") for name, score in predictions.items()}}\n        write_json(output / "metrics_before_reranker.json", results)\n        for name, values in predictions.items():\n            pairs[name + "_pref"] = values[:, 0]\n            pairs[name + "_qual"] = values[:, 1]\n        pairs.loc[pairs.split.eq("test")].to_csv(output / "test_predictions.csv", index=False)\n        stage = "qualification_reranker"\n        results["reranker"] = rerank_qualification(pairs, texts, cosine, output)\n        results["total_seconds"] = time.monotonic() - started\n        write_json(output / "metrics.json", results)\n        write_report(results, output)\n        write_json(output / "status.json", {"status": "complete", "elapsed_seconds": results["total_seconds"]})\n        LOGGER.info("EXPERIMENT_COMPLETE %s", json.dumps(results))\n    except Exception as error:\n        write_json(output / "status.json", {"status": "failed", "stage": stage,\n                   "error_type": type(error).__name__, "error": str(error), "elapsed_seconds": time.monotonic() - started})\n        LOGGER.exception("Experiment failed at stage=%s; no automatic fallback", stage)\n        raise\n\n\nif __name__ == "__main__":\n    main()\n'
module = types.ModuleType('jobrec_v1')
exec(compile(source, 'jobrec_v1.py', 'exec'), module.__dict__)
sys.modules['jobrec_v1'] = module


In [ ]:
"""Responsibilities: register a frozen experiment module.
Implementation: compile source in an isolated module namespace.
Related Modules: later notebook cells import this registered module.
Declaration Index: None.
Variable Index: source is embedded Python; module is the registered namespace.
"""
import sys
import types
source = '"""职责：在固定 v1 数据与向量上验证特征和任务共享的贡献。\n实现：校验输入哈希；同一起点比较四种网络；按查询配对 bootstrap 测试预测。\n关联：jobrec_v1 是 notebook 内封装的原版 experiment.py，源码 SHA256 固定。\n目录：verify_inputs、feature_view、IndependentHeads、IndependentHeads.__init__、\nIndependentHeads.forward、masked_loss、train_variant、bootstrap_ap、query_bootstrap、main。\n变量索引：CONFIG 为原实验配置；PROTOCOL 为新增消融配置；INPUT_HASHES 固定输入哈希；\nLOGGER 为不含凭据/档案正文的阶段日志。实例 branches 存两个互不共享参数的网络。\n约束：不修改 v1，不更换划分、训练种子或超参数；bootstrap 不代表训练随机性检验。\n"""\n\nfrom __future__ import annotations\n\nimport copy\nimport gc\nimport hashlib\nimport json\nimport logging\nimport random\nimport sys\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import average_precision_score\n\nimport jobrec_v1 as base\n\nCONFIG = base.CONFIG\nPROTOCOL = {\n    "name": "jobrec-fixed-split-ablation-v2",\n    "variants": ["shared_full", "shared_structured", "shared_vector", "independent_full"],\n    "bootstrap_replicates": 2000,\n    "bootstrap_seed": 42,\n    "initialization": "reset seed42 immediately before every variant; clone identical shared trunk for independent branches",\n    "feature_ablation": "zero removed blocks, retaining 2059 inputs and identical shared parameter counts",\n    "checkpoint_selection": "same joint mean validation AP for all four variants",\n    "comparison_scope": "v2-to-v2 controlled comparison; v1 RNG position before network initialization may differ",\n}\nINPUT_HASHES = {\n    "config.json": "8dea69e2e61e7b5ce0812072f04eb17f931aad0c2fa97aeba2deb88e8db28513",\n    "data_manifest.json": "9f692a0f8921625287408c9cbd5343823a2c612e50a43db0937667b6d0c73957",\n    "pair_splits.csv": "37b134a5a2461481ce9d04318c6d05bf5025752173236b334db97cf130db63c0",\n    "split_manifest.json": "dd228b8394d9d14e0bca0c77fadb07625d7f7ff3ee07472a2a93ed083cfd7eb1",\n    "embeddings.npz": "c468440b592455f1503b022784c7ab44a7c4b9600ffe4a17e3f46f462ec2cc3b",\n    "test_predictions.csv": "2045a94220c1d61bb19a3de4da26d10fe74eab5853d048e9eeb3bbb23b0b660b",\n    "feature_scaler.json": "4ec782a0009581b88c7bcd21d824c410181350916cbda163b1680b31f53a5051",\n}\nLOGGER = logging.getLogger("jobrec_ablation")\n\n\ndef verify_inputs(folder: Path) -> None:\n    """功能：验证父实验产物身份；输入父目录；输出无，失败抛出异常。\n    逻辑：逐文件比对 SHA256 并检查配置；约束：不接受同名但已更新的父产物。\n    """\n    for name, expected in INPUT_HASHES.items():\n        actual = hashlib.sha256((folder / name).read_bytes()).hexdigest()\n        if actual != expected:\n            raise ValueError(f"Parent artifact changed: {name}: {actual}")\n    if json.loads((folder / "config.json").read_text()) != CONFIG:\n        raise ValueError("Parent config differs from embedded v1 source")\n    LOGGER.info("Verified all %d parent artifact hashes", len(INPUT_HASHES))\n\n\ndef feature_view(features: np.ndarray, variant: str) -> np.ndarray:\n    """功能：移除指定特征块；输入 N×2059矩阵和实验名；输出独立副本。\n    逻辑：结构化组把前2048维归零，向量组把后11维归零；约束：原矩阵保持不变，\n    保留相同层宽与参数量，但零输入意味着实际可用特征容量仍不同。\n    """\n    if features.shape[1] != 2059 or variant not in PROTOCOL["variants"]:\n        raise ValueError("Unexpected feature dimension or variant")\n    result = features.copy()\n    if variant == "shared_structured":\n        result[:, :-11] = 0\n    elif variant == "shared_vector":\n        result[:, -11:] = 0\n    return result\n\n\nclass IndependentHeads(torch.nn.Module):\n    """功能：将两任务拆为独立网络；逻辑：各自克隆同一初始 trunk 和对应 head。\n    约束：每任务层宽不变，总参数约翻倍；这是共享与不共享对照，不是等总参数对照。\n    """\n\n    def __init__(self, template: base.DualHeadModel):\n        """功能：创建独立分支；输入已初始化的共享模型；输出模块实例。\n        逻辑：deepcopy 阻断参数别名且不消耗新的初始化随机数；约束：仅 branches 有状态。\n        """\n        super().__init__()\n        self.branches = torch.nn.ModuleList([\n            torch.nn.Sequential(copy.deepcopy(template.trunk), copy.deepcopy(head))\n            for head in template.heads\n        ])\n\n    def forward(self, features: torch.Tensor) -> torch.Tensor:\n        """功能：计算独立任务 logits；输入 B×2059；输出 B×2。\n        逻辑：拼接两个分支输出；约束：训练时两分支各自抽取 dropout 掩码。\n        """\n        return torch.cat([branch(features) for branch in self.branches], dim=1)\n\n\ndef masked_loss(logits: torch.Tensor, labels: torch.Tensor, observed: torch.Tensor,\n                positive_weights: torch.Tensor) -> torch.Tensor:\n    """功能：仅使用观测标签计算损失；输入 B×2 logits/标签/mask 及2维正类权重。\n    输出标量；逻辑：各有观测任务内部取均值，再对任务取均值，与 v1 相同。\n    约束：未知位置标签必须已填有限占位值；两任务全无观测时明确报错。\n    """\n    element = torch.nn.functional.binary_cross_entropy_with_logits(\n        logits, labels, pos_weight=positive_weights, reduction="none")\n    losses = [(element[:, c] * observed[:, c]).sum() / observed[:, c].sum()\n              for c in range(2) if observed[:, c].any()]\n    if not losses:\n        raise ValueError("Batch has no observed task labels")\n    return torch.stack(losses).mean()\n\n\ndef train_variant(pairs: pd.DataFrame, features: np.ndarray, variant: str, output: Path) -> np.ndarray:\n    """功能：执行一次受控训练；输入配对/特征/组名/目录；输出所有配对的两任务分数。\n    逻辑：每组训练前重设相同seed；同 minibatch 顺序、损失和联合验证选择；保存检查点。\n    约束：仅在12轮内选择，不使用测试标签选模型；v1训练前的RNG位置不作为本组对照。\n    """\n    random.seed(CONFIG["seed"])\n    np.random.seed(CONFIG["seed"])\n    torch.manual_seed(CONFIG["seed"])\n    torch.cuda.manual_seed_all(CONFIG["seed"])\n    template = base.DualHeadModel(features.shape[1])\n    model = (IndependentHeads(template) if variant == "independent_full" else template).to("cuda:0")\n    parameter_count = sum(p.numel() for p in model.parameters())\n    initial = {k: value.cpu() for k, value in template.state_dict().items()}\n    torch.save(initial, output / "initial_shared_template.pt")\n    x = torch.from_numpy(feature_view(features, variant)).to("cuda:0")\n    targets = pairs[["label_pref", "label_qual"]].to_numpy(dtype=np.float32)\n    observed = np.isfinite(targets)\n    y = torch.from_numpy(np.nan_to_num(targets, nan=0)).to("cuda:0")\n    mask = torch.from_numpy(observed).to("cuda:0")\n    train_ids = np.flatnonzero(pairs.split.eq("train"))\n    val_ids = np.flatnonzero(pairs.split.eq("validation"))\n    weights = []\n    for column in range(2):\n        labels = targets[train_ids, column]\n        labels = labels[np.isfinite(labels)]\n        weights.append(float((labels == 0).sum() / (labels == 1).sum()))\n    positive_weights = torch.tensor(weights, device="cuda:0")\n    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["learning_rate"], weight_decay=CONFIG["weight_decay"])\n    best_score, best_state, history = -1.0, None, []\n    rng = np.random.default_rng(CONFIG["seed"])\n    started = time.monotonic()\n    for epoch in range(1, CONFIG["epochs"] + 1):\n        model.train()\n        losses, order = [], rng.permutation(train_ids)\n        for start in range(0, len(order), CONFIG["training_batch_size"]):\n            ids = order[start:start + CONFIG["training_batch_size"]]\n            optimizer.zero_grad(set_to_none=True)\n            loss = masked_loss(model(x[ids]), y[ids], mask[ids], positive_weights)\n            loss.backward()\n            optimizer.step()\n            losses.append(float(loss.item()))\n        model.eval()\n        with torch.inference_mode():\n            val_scores = torch.sigmoid(model(x[val_ids])).cpu().numpy()\n        aps = [float(average_precision_score(targets[val_ids, c][observed[val_ids, c]],\n               val_scores[:, c][observed[val_ids, c]])) for c in range(2)]\n        validation_mean = float(np.mean(aps))\n        if not np.isfinite(validation_mean):\n            raise ValueError("Non-finite validation AP")\n        row = {"epoch": epoch, "mean_batch_loss": float(np.mean(losses)), "validation_ap": aps}\n        history.append(row)\n        LOGGER.info("%s epoch=%s", variant, row)\n        if validation_mean > best_score:\n            best_score, best_state, selected_epoch = validation_mean, copy.deepcopy(model.state_dict()), epoch\n    model.load_state_dict(best_state)\n    with torch.inference_mode():\n        scores = torch.sigmoid(model(x)).cpu().numpy()\n    if not np.isfinite(scores).all():\n        raise ValueError("Non-finite prediction")\n    torch.save({"state_dict": {k: v.cpu() for k, v in best_state.items()}, "variant": variant,\n                "config": CONFIG, "protocol": PROTOCOL}, output / "checkpoint.pt")\n    base.write_json(output / "training_history.json", {"epochs": history, "selected_epoch": selected_epoch,\n                    "validation_mean_ap": best_score, "parameter_count": parameter_count,\n                    "positive_weights": weights, "training_seconds": time.monotonic() - started})\n    del model, template, x, y, mask, optimizer, best_state\n    gc.collect()\n    torch.cuda.empty_cache()\n    return scores\n\n\ndef bootstrap_ap(labels: np.ndarray, scores: np.ndarray, group_ids: np.ndarray,\n                 counts: np.ndarray) -> np.ndarray:\n    """功能：计算重复抽样查询后的配对AP；输入标签/分数/组索引/每次组抽样计数。\n    输出每次抽样AP；逻辑：查询重复数作为样本权重，按相同分数整体计算precision阶梯。\n    约束：不是query平均AP；拒绝无正例抽样，按100个replicate分块限制内存。\n    """\n    order = np.argsort(-scores, kind="stable")\n    sorted_labels, sorted_scores = labels[order], scores[order]\n    ends = np.r_[np.flatnonzero(np.diff(sorted_scores)), len(scores) - 1]\n    values = []\n    for start in range(0, len(counts), 100):\n        weights = counts[start:start + 100, group_ids[order]].astype(float)\n        positive = np.cumsum(weights * sorted_labels, axis=1)[:, ends]\n        total = np.cumsum(weights, axis=1)[:, ends]\n        if (positive[:, -1] == 0).any():\n            raise ValueError("Bootstrap replicate has no positives")\n        precision = np.divide(positive, total, out=np.zeros_like(positive), where=total > 0)\n        increments = np.diff(np.column_stack([np.zeros(len(positive)), positive]), axis=1)\n        values.extend(((increments * precision).sum(axis=1) / positive[:, -1]).tolist())\n    return np.asarray(values)\n\n\ndef query_bootstrap(predictions: pd.DataFrame, model_names: list[str], output: Path) -> dict:\n    """功能：量化固定模型在测试查询抽样下的不确定性；输入测试预测/模型名/目录；输出区间。\n    逻辑：偏好按候选人、资格按岗位抽样2000次，同一任务所有模型共用抽样；输出AP、\n    nDCG@5和Recall@5及预先指定的配对差值区间。\n    约束：百分位95%区间是探索性未校正多重比较；偏好结果条件于当前测试岗位集合，\n    不覆盖训练种子或换岗位划分的方差；全负查询只从排名均值排除。\n    """\n    result = {"replicates": PROTOCOL["bootstrap_replicates"], "models": {}, "comparisons": {}}\n    contrasts = [("shared_full", name) for name in (\n        "shared_structured", "shared_vector", "independent_full", "structured_logistic")]\n    for task in ("pref", "qual"):\n        frame = predictions.loc[predictions["label_" + task].notna()].reset_index(drop=True)\n        group_column = "user" if task == "pref" else "item"\n        groups, ids = np.unique(frame[group_column], return_inverse=True)\n        labels = frame["label_" + task].to_numpy()\n        rng = np.random.default_rng(PROTOCOL["bootstrap_seed"])\n        counts = rng.multinomial(len(groups), np.full(len(groups), 1 / len(groups)),\n                                 size=PROTOCOL["bootstrap_replicates"])\n        distributions = {}\n        for name in model_names:\n            scores = frame[name + "_" + task].to_numpy()\n            ap = bootstrap_ap(labels, scores, ids, counts)\n            query_scores = np.full((len(groups), 2), np.nan)\n            for i in range(len(groups)):\n                group = frame.loc[ids == i]\n                if group["label_" + task].sum() > 0:\n                    metrics = base.ranking_metrics(group, scores[ids == i], task)\n                    query_scores[i] = [metrics["ndcg@5"], metrics["recall@5"]]\n            valid = np.isfinite(query_scores[:, 0])\n            denominator = counts[:, valid].sum(axis=1)\n            if (denominator == 0).any():\n                raise ValueError("Bootstrap replicate has no positive queries")\n            rank = counts[:, valid] @ query_scores[valid] / denominator[:, None]\n            distributions[name] = {"average_precision": ap, "ndcg@5": rank[:, 0], "recall@5": rank[:, 1]}\n            result["models"].setdefault(name, {})[task] = {\n                metric: np.quantile(values, [0.025, 0.975]).tolist()\n                for metric, values in distributions[name].items()\n            }\n            LOGGER.info("Bootstrap completed task=%s model=%s queries=%d", task, name, len(groups))\n        for left, right in contrasts:\n            key = left + "_minus_" + right\n            result["comparisons"].setdefault(key, {})[task] = {\n                metric: {"ci95": np.quantile(distributions[left][metric] - distributions[right][metric],\n                                             [0.025, 0.975]).tolist()}\n                for metric in distributions[left]\n            }\n    base.write_json(output / "bootstrap.json", result)\n    return result\n\n\ndef main() -> None:\n    """功能：执行消融与结果审计；输入挂载的v1产物和Kaggle GPU环境；输出ablation_v2目录。\n    逻辑：先验证身份及重新生成的配对/特征，再四组训练和统一评测，最后不确定性分析。\n    约束：任何失败记录阶段并抛出；不改超参数、重试或回退。模型下载/编码直接复用已核验向量。\n    """\n    output = Path("/kaggle/working/ablation_v2")\n    output.mkdir(parents=True, exist_ok=True)\n    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",\n                        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(output / "execution.log")])\n    stage, started = "parent_verification", time.monotonic()\n    base.write_json(output / "protocol.json", {"base_config": CONFIG, "ablation": PROTOCOL, "input_hashes": INPUT_HASHES})\n    try:\n        candidates = list(Path("/kaggle/input").rglob("jobrec_v1/config.json"))\n        if len(candidates) != 1:\n            raise ValueError(f"Expected one mounted parent artifact, found {len(candidates)}")\n        parent = candidates[0].parent\n        verify_inputs(parent)\n        stage = "environment_and_data"\n        base.bootstrap(output)\n        frames = base.download_data(output)\n        if json.loads((output / "data_manifest.json").read_text()) != json.loads((parent / "data_manifest.json").read_text()):\n            raise ValueError("Downloaded source data changed")\n        pairs, _, _, _, structured = base.prepare_data(frames, output)\n        pd.testing.assert_frame_equal(pairs, pd.read_csv(parent / "pair_splits.csv"), check_dtype=False)\n        archived = np.load(parent / "embeddings.npz")\n        features, _ = base.build_pair_features(pairs, (archived["candidates"], archived["jobs"]), structured, output)\n        if json.loads((output / "feature_scaler.json").read_text()) != json.loads((parent / "feature_scaler.json").read_text()):\n            raise ValueError("Feature preprocessing differs from v1")\n        results, scores = {}, {}\n        for variant in PROTOCOL["variants"]:\n            stage = "training_" + variant\n            destination = output / variant\n            destination.mkdir()\n            scores[variant] = train_variant(pairs, features, variant, destination)\n        stage = "evaluation"\n        test = pairs.loc[pairs.split.eq("test")].copy()\n        for name, values in scores.items():\n            results[name] = base.evaluate_scores(pairs, values, "test")\n            test[name + "_pref"] = values[test.index, 0]\n            test[name + "_qual"] = values[test.index, 1]\n        previous = pd.read_csv(parent / "test_predictions.csv")\n        pd.testing.assert_frame_equal(test[["user", "item"]].reset_index(drop=True), previous[["user", "item"]])\n        for task in ("pref", "qual"):\n            test["structured_logistic_" + task] = previous["structured_logistic_" + task].to_numpy()\n        results["structured_logistic"] = base.evaluate_scores(test,\n            test[["structured_logistic_pref", "structured_logistic_qual"]].to_numpy(), "test")\n        test.to_csv(output / "test_predictions.csv", index=False)\n        base.write_json(output / "metrics.json", results)\n        stage = "query_bootstrap"\n        query_bootstrap(test, list(results), output)\n        base.write_json(output / "status.json", {"status": "complete", "total_seconds": time.monotonic() - started})\n        LOGGER.info("ABLATION_COMPLETE %s", json.dumps(results))\n    except Exception as error:\n        base.write_json(output / "status.json", {"status": "failed", "stage": stage,\n                        "error_type": type(error).__name__, "error": str(error)})\n        LOGGER.exception("Ablation failed at stage=%s; no fallback", stage)\n        raise\n\n\nif __name__ == "__main__":\n    main()\n'
module = types.ModuleType('jobrec_v2')
exec(compile(source, 'jobrec_v2.py', 'exec'), module.__dict__)
sys.modules['jobrec_v2'] = module


In [ ]:
"""Responsibilities: register a frozen experiment module.
Implementation: compile source in an isolated module namespace.
Related Modules: later notebook cells import this registered module.
Declaration Index: None.
Variable Index: source is embedded Python; module is the registered namespace.
"""
import sys
import types
source = '"""职责：定义v4冻结协议、实体级资料遮蔽与缺失感知特征。\n实现：SHA256确定掩码；先隐藏原始字段，再计算NaN感知的11项配对特征。\n关联：jobrec_v1提供既定列表解析和特征名；experiment负责训练，evaluation负责评测。\n目录：uniform、scenarios、make_masks、hide_profiles、unknown、number、sequence、\n      academic、boolean、features、feature_view。\n变量索引：USER_BLOCKS/JOB_BLOCKS为资料块；OPTIONAL/CORE为列索引；LEVELS为学业编码；\n      RANKER_CONFIG为历史树参数；PROTOCOL为已批准的新条件；HISTORICAL为v3产物哈希。\n约束：不读取标签决定缺失；不补零、不推断隐藏字段、不自动重试；ID不是预测特征。\n"""\n\nimport hashlib\nimport json\n\nimport numpy as np\nimport pandas as pd\n\nimport jobrec_v1 as base\n\nUSER_BLOCKS = [("skills",), ("interests",), ("masters_major", "second_major", "undergrad_major"),\n               ("in_person_commitment",), ("gpa",), ("months_experience",), ("academic_level",),\n               ("hours_per_week",), ("length_of_commitment",), ("num_publications",), ("commit_to_summer",)]\nJOB_BLOCKS = [(x,) for x in ("required_skills", "industry", "acceptable_majors", "job_in_person_commitment",\n              "min_gpa", "min_months_experience", "min_academic_level", "min_hours_per_week",\n              "min_length_of_commitment", "min_num_publications")]\nOPTIONAL = [4, 7, 8, 9, 10]\nCORE = [0, 1, 2, 3, 5, 6]\nLEVELS = {"UG1": 1, "UG2": 2, "UG3": 3, "UG4": 4, "MS1": 5, "MS2": 6,\n          "PhD1": 7, "PhD2": 8, "PhD3": 9, "PhD4": 10, "PhD5": 11}\nRANKER_CONFIG = {"objective": "lambdarank", "n_estimators": 200, "learning_rate": 0.05,\n                 "num_leaves": 15, "min_child_samples": 20, "reg_lambda": 1.0,\n                 "random_state": 42, "n_jobs": 2, "verbosity": -1,\n                 "deterministic": True, "force_col_wise": True}\nPROTOCOL = {"name": "jobrec-missing-v4", "seed": 42, "mask_replicates": 5,\n            "training_mix": {"clean": 0.5, "random_30": 0.25, "optional_bundle": 0.25},\n            "validation_scenarios": ["clean", "random_30", "optional_bundle"],\n            "primary_scenarios": ["random_30", "optional_bundle", "mixed_optional"],\n            "selection": "per-task equal mean validation AP; grid20..200; earliest tie",\n            "models": ["H", "A", "B", "C", "D"], "tree_grid": list(range(20, 201, 20)),\n            "bootstrap_replicates": 2000, "clean_max_drop": 0.02, "missing_min_gain": 0.03,\n            "ranker": RANKER_CONFIG, "user_blocks": USER_BLOCKS, "job_blocks": JOB_BLOCKS,\n            "optional_columns": OPTIONAL, "core_columns": CORE,\n            "feature_names": list(base.STRUCTURED_NAMES), "lightgbm": "4.6.0",\n            "scope": "development diagnostics; synthetic labels; no interview fusion or deployment"}\nHISTORICAL = {\n    "lambdarank/pref.txt": "7cf96ed020d367e9798893ffe7713674620073bda0941a7e1e8682758cb2d8cb",\n    "lambdarank/qual.txt": "b0c310f4dcb66c375794fbe2f931a19ca108ab17709d64ef0fc6d0daca9b00e7",\n    "test_predictions.csv": "1eff7faac87ec5011f70c247898935d7569169d3dcc28337fc3d1aa5d4431f62",\n    "metrics.json": "577b26e0b871adbbf7f3e98806b5e862a805e3d95178c437be28ff6b7a676a2d",\n}\n\n\ndef uniform(namespace: str, scenario: str, repeat: int, entity: str, block: str) -> float:\n    """功能：生成稳定均匀值；输入用途/场景/重复/实体/块；输出[0,1)数。\n    逻辑：seed42和参数JSON的SHA256前8字节转无符号数；约束：不依赖行序或进程hash。\n    """\n    raw = json.dumps([42, namespace, scenario, repeat, str(entity), block], separators=(",", ":")).encode()\n    return int.from_bytes(hashlib.sha256(raw).digest()[:8], "big") / 2**64\n\n\ndef scenarios() -> list[tuple[str, int]]:\n    """功能：列举固定评测清单；输入模块协议；输出场景/重复序列。\n    逻辑：确定性场景一次，五种随机场景各五次；约束：r0主报告，不按成绩选择重复。\n    """\n    fixed = ["clean", "optional_bundle", "skills_only", "no_candidate_information"]\n    fixed += ["single_" + str(i) for i in range(11)]\n    random = ["random_10", "random_30", "random_50", "mixed_optional", "job_random_30"]\n    return [(name, 0) for name in fixed] + [(name, r) for name in random for r in range(5)]\n\n\ndef make_masks(users: pd.DataFrame, jobs: pd.DataFrame, scenario: str, repeat: int = 0,\n               namespace: str = "evaluation") -> tuple[np.ndarray, np.ndarray, list[str]]:\n    """功能：生成实体资料块缺失掩码；输入含ID的实体表/场景/重复/用途；输出两掩码及用户分配。\n    逻辑：固定或SHA抽样选择块，training_mix先分配50/25/25类别；约束：只读ID，不读标签。\n    """\n    allowed = {name for name, _ in scenarios()} | {"training_mix"}\n    if scenario not in allowed or repeat not in range(5):\n        raise ValueError("Unregistered mask scenario or repeat")\n    um = np.zeros((len(users), 11), dtype=bool)\n    jm = np.zeros((len(jobs), 10), dtype=bool)\n    assignments = []\n    for row, entity in enumerate(users.student_id):\n        mode = scenario\n        if scenario == "training_mix":\n            choice = uniform(namespace, scenario, repeat, entity, "assignment")\n            mode = "clean" if choice < .5 else "random_30" if choice < .75 else "optional_bundle"\n        assignments.append(mode)\n        if mode.startswith("single_"):\n            um[row, int(mode.split("_")[1])] = True\n        elif mode == "optional_bundle":\n            um[row, OPTIONAL] = True\n        elif mode == "mixed_optional":\n            if uniform(namespace, scenario, repeat, entity, "assignment") < .5:\n                um[row, OPTIONAL] = True\n        elif mode == "skills_only":\n            um[row, 1:] = True\n        elif mode == "no_candidate_information":\n            um[row] = True\n        elif mode.startswith("random_"):\n            probability = int(mode.split("_")[1]) / 100\n            um[row] = [uniform(namespace, scenario, repeat, entity, "/".join(block)) < probability\n                       for block in USER_BLOCKS]\n    if scenario == "job_random_30":\n        for row, entity in enumerate(jobs.job_id):\n            jm[row] = [uniform(namespace, scenario, repeat, entity, "/".join(block)) < .3\n                       for block in JOB_BLOCKS]\n    return um, jm, assignments\n\n\ndef hide_profiles(frame: pd.DataFrame, masks: np.ndarray, blocks: list) -> pd.DataFrame:\n    """功能：物理隐藏原始字段；输入实体表/块掩码/块定义；输出独立资料表。\n    逻辑：相关列转object后将选择值置None；约束：不修改源表，不保留隐藏值供特征计算。\n    """\n    if masks.shape != (len(frame), len(blocks)):\n        raise ValueError("Mask shape differs from entity table")\n    result = frame.copy()\n    for col, block in enumerate(blocks):\n        for name in block:\n            result[name] = result[name].astype(object)\n            result.loc[masks[:, col], name] = None\n    return result\n\n\ndef unknown(value: object) -> bool:\n    """功能：判定标量未知；输入单字段值；输出布尔；逻辑：仅None或浮点NaN为未知。\n    约束：空字符串、无穷大、错误类型不伪装成缺失，列表由sequence单独解析。\n    """\n    return value is None or isinstance(value, (float, np.floating)) and np.isnan(value)\n\n\ndef number(value: object) -> float:\n    """功能：解析已知数值；输入标量；输出float或NaN；逻辑：未知保留，其他转float。\n    约束：无穷大、布尔和非法文本报错，已知零不改写。\n    """\n    if unknown(value):\n        return np.nan\n    if isinstance(value, (bool, np.bool_)):\n        raise ValueError("Boolean is not a numeric quantity")\n    result = float(value)\n    if not np.isfinite(result):\n        raise ValueError("Invalid non-finite numeric quantity")\n    return result\n\n\ndef sequence(value: object) -> set | None:\n    """功能：解析技能/兴趣/专业列表；输入字段；输出集合或None。\n    逻辑：已知字段沿用v1解析器；约束：未知不会变为空集合，不调整大小写。\n    """\n    return None if unknown(value) else set(base.parse_list(value))\n\n\ndef academic(value: object) -> float:\n    """功能：学业阶段映射；输入代码；输出原编码或NaN；逻辑：查固定LEVELS。\n    约束：未知代码报错，不将未支持代码视为普通缺失。\n    """\n    return np.nan if unknown(value) else float(LEVELS[value])\n\n\ndef boolean(value: object) -> float:\n    """功能：暑期布尔转换；输入值；输出0/1/NaN；逻辑：只接受true/false字符串或布尔。\n    约束：未知不映射false，错误字符串报错；原有效数据的含义保持不变。\n    """\n    if unknown(value):\n        return np.nan\n    text = str(value).lower()\n    if text not in ("true", "false"):\n        raise ValueError("Invalid commitment boolean")\n    return float(text == "true")\n\n\ndef features(pairs: pd.DataFrame, users: pd.DataFrame, jobs: pd.DataFrame) -> np.ndarray:\n    """功能：重算缺失感知匹配；输入配对索引和已隐藏原始资料；输出N×11 float32。\n    逻辑：解析实体一次，集合匹配逐配对，数值差按索引向量化；约束：不访问完整资料缓存。\n    专业以已知专业集合为依据，原无第二/硕士专业不使本科信息失效；整个专业块未知才NaN。\n    输入u_idx/j_idx必须对应给定实体表；空岗位技能沿用历史报错；NaN允许，无穷大拒绝。\n    """\n    ui, ji = pairs.u_idx.to_numpy(), pairs.j_idx.to_numpy()\n    us = [sequence(v) for v in users.skills]\n    req = [sequence(v) for v in jobs.required_skills]\n    if any(v is not None and not v for v in req):\n        raise ValueError("Empty required skill list is outside the frozen protocol")\n    interests = [sequence(v) for v in users.interests]\n    majors = [{v for v in row if not unknown(v)} or None\n              for row in users[["masters_major", "second_major", "undergrad_major"]].itertuples(index=False, name=None)]\n    accepted = [sequence(v) for v in jobs.acceptable_majors]\n    industries = jobs.industry.tolist()\n    modes_u, modes_j = users.in_person_commitment.tolist(), jobs.job_in_person_commitment.tolist()\n    result = np.full((len(pairs), 11), np.nan, dtype=np.float32)\n    for k, (u, j) in enumerate(zip(ui, ji)):\n        if us[u] is not None and req[j] is not None:\n            result[k, 0] = len(us[u] & req[j]) / len(req[j])\n        if interests[u] is not None and not unknown(industries[j]):\n            result[k, 1] = float(industries[j] in interests[u])\n        if majors[u] is not None and accepted[j] is not None:\n            result[k, 2] = float(bool(majors[u] & accepted[j]))\n        if not unknown(modes_u[u]) and not unknown(modes_j[j]):\n            result[k, 3] = float(modes_u[u] == modes_j[j])\n    for c in range(4, 10):\n        parser = academic if c == 6 else number\n        uv = np.asarray([parser(v) for v in users[USER_BLOCKS[c][0]]])\n        jv = np.asarray([parser(v) for v in jobs[JOB_BLOCKS[c][0]]])\n        result[:, c] = uv[ui] - jv[ji]\n    result[:, 10] = np.asarray([boolean(v) for v in users.commit_to_summer])[ui]\n    if np.isinf(result).any():\n        raise ValueError("Infinite feature")\n    return result\n\n\ndef feature_view(matrix: np.ndarray, model: str) -> np.ndarray:\n    """功能：提供模型输入视图；输入N×11数组/模型名；输出11/22/6维数组。\n    逻辑：C添加NaN指示，D取固定六列，其余复制；约束：拒绝未知模型、无穷大和维度错。\n    """\n    if model not in PROTOCOL["models"] or matrix.ndim != 2 or matrix.shape[1] != 11 or np.isinf(matrix).any():\n        raise ValueError("Invalid model or structured feature matrix")\n    if model == "C":\n        return np.concatenate([matrix, np.isnan(matrix).astype(np.float32)], axis=1)\n    return matrix[:, CORE].copy() if model == "D" else matrix.copy()\n'
module = types.ModuleType('jobrec_missing')
exec(compile(source, 'jobrec_missing.py', 'exec'), module.__dict__)
sys.modules['jobrec_missing'] = module


In [ ]:
"""Responsibilities: register a frozen experiment module.
Implementation: compile source in an isolated module namespace.
Related Modules: later notebook cells import this registered module.
Declaration Index: None.
Variable Index: source is embedded Python; module is the registered namespace.
"""
import sys
import types
source = '"""职责：评估v4固定预测并保存查询级证据、区间和工程候选门槛。\n实现：沿用v1主指标；显式记录并列与正例排名；复用v2加权AP进行配对查询bootstrap。\n关联：jobrec_missing定义协议；experiment和verify调用analyze，不在此训练或选择模型。\n目录：query_details、summarize_strata、analyze。\n变量索引：LOGGER为阶段日志；CONTRASTS为预登记模型差值；实例状态无。\n约束：同任务跨模型/场景共用重采样；全负查询只从排名均值排除；诊断不证明公平性。\n"""\n\nimport logging\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nimport jobrec_v1 as base\nimport jobrec_v2 as previous\nimport jobrec_missing as spec\n\nLOGGER = logging.getLogger("jobrec_v4_evaluation")\nCONTRASTS = [("B", "A"), ("C", "B"), ("D", "B")] + [(m, "H") for m in "ABCD"]\n\n\ndef query_details(frame: pd.DataFrame, scores: np.ndarray, task: str,\n                  available: np.ndarray) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """功能：记录每查询排名及正例位置；输入仅该任务已观察行/分数/任务/资料可用数。\n    输出查询表和正例表；逻辑：稳定降序保持历史并列口径，k截断与历史一致。\n    约束：记录并列率；全负查询指标为NaN保留在表中，排序位置为1起始且依赖同分行序。\n    """\n    frame = frame.reset_index(drop=True)\n    rows, positives = [], []\n    for query, indices in frame.groupby("user" if task == "pref" else "item", sort=True).indices.items():\n        indices = np.asarray(indices)\n        labels = frame.iloc[indices]["label_" + task].to_numpy()\n        values = scores[indices]\n        order = np.argsort(-values, kind="stable")\n        ranked = labels[order]\n        count = int(labels.sum())\n        row = {"query": query, "pool": len(indices), "positives": count,\n               "tied_fraction": float(1 - len(np.unique(values)) / len(values))}\n        for k in (5, 10):\n            limit = min(k, len(order))\n            ideal = (1 / np.log2(np.arange(min(count, limit)) + 2)).sum()\n            row[f"ndcg@{k}"] = float((ranked[:limit] / np.log2(np.arange(limit) + 2)).sum() / ideal) if count else np.nan\n            row[f"recall@{k}"] = float(ranked[:limit].sum() / count) if count else np.nan\n        rows.append(row)\n        for rank, local in enumerate(order, 1):\n            if labels[local] == 1:\n                index = indices[local]\n                positives.append({"user": frame.iloc[index].user, "item": frame.iloc[index].item,\n                                  "query": query, "rank": rank, "hit5": int(rank <= 5),\n                                  "available": int(available[index])})\n    return pd.DataFrame(rows), pd.DataFrame(positives)\n\n\ndef summarize_strata(positive: pd.DataFrame, clean: pd.DataFrame) -> list[dict]:\n    """功能：对同一正例比较遮蔽前后排名；输入某场景与clean的正例表；输出按可用数分层摘要。\n    逻辑：按user/item一一合并后汇总人数/正例数/排名变化/命中变化；约束：候选池不删减。\n    """\n    joined = positive.merge(clean[["user", "item", "rank", "hit5"]], on=["user", "item"],\n                            suffixes=("", "_clean"), validate="one_to_one")\n    output = []\n    for count, part in joined.groupby("available"):\n        output.append({"available": int(count), "users": int(part.user.nunique()), "positive_pairs": len(part),\n                       "mean_rank": float(part["rank"].mean()),\n                       "mean_rank_worsening": float((part["rank"] - part.rank_clean).mean()),\n                       "hit5": float(part.hit5.mean()), "hit5_change": float((part.hit5 - part.hit5_clean).mean())})\n    return output\n\n\ndef analyze(pairs: pd.DataFrame, predictions: dict, feature_arrays: dict, output: Path) -> dict:\n    """功能：完整分析下载或远端固定预测；输入测试配对/场景模型预测/场景特征/输出目录。\n    输出metrics字典并保存查询证据/区间/门槛；逻辑：逐场景主指标，r0共享2000次查询抽样。\n    约束：无训练或选模；预测须有限；r1-r4仅敏感性分析；工程门槛不等于上线许可。\n    """\n    output.mkdir(parents=True, exist_ok=True)\n    metrics, queries, positive_tables, all_query_rows, all_positive_rows = {}, {}, {}, [], []\n    for scene, _ in feature_arrays.items():\n        available = np.isfinite(feature_arrays[scene]).sum(axis=1)\n        metrics[scene] = {}\n        for model in spec.PROTOCOL["models"]:\n            scores = predictions[scene + "__" + model]\n            if scores.shape != (len(pairs), 2) or not np.isfinite(scores).all():\n                raise ValueError(f"Invalid predictions: {scene}/{model}")\n            metrics[scene][model] = base.evaluate_scores(pairs, scores, "test")\n            for column, task in enumerate(("pref", "qual")):\n                observed = pairs["label_" + task].notna().to_numpy()\n                q, p = query_details(pairs.loc[observed], scores[observed, column], task, available[observed])\n                key = scene + "__" + model + "__" + task\n                queries[key], positive_tables[key] = q, p\n                all_query_rows.append(q.assign(scene=scene, model=model, task=task))\n                all_positive_rows.append(p.assign(scene=scene, model=model, task=task))\n                for metric in ("ndcg@5", "ndcg@10", "recall@5", "recall@10"):\n                    np.testing.assert_allclose(q[metric].mean(), metrics[scene][model][task][metric], rtol=0, atol=1e-12)\n                metrics[scene][model][task]["mean_tied_fraction"] = float(q.tied_fraction.mean())\n                metrics[scene][model][task]["queries_with_ties"] = int(q.tied_fraction.gt(0).sum())\n        LOGGER.info("METRICS %s completed", scene)\n    base.write_json(output / "metrics.json", metrics)\n    pd.concat(all_query_rows, ignore_index=True).to_csv(output / "query_metrics.csv.gz", index=False)\n    pd.concat(all_positive_rows, ignore_index=True).to_csv(output / "positive_ranks.csv.gz", index=False)\n    strata = {}\n    for scene in ("random_30_r0", "mixed_optional_r0", "optional_bundle_r0", "skills_only_r0"):\n        for model in spec.PROTOCOL["models"]:\n            for task in ("pref", "qual"):\n                key = scene + "__" + model + "__" + task\n                strata[key] = summarize_strata(positive_tables[key], positive_tables["clean_r0__" + model + "__" + task])\n    base.write_json(output / "completeness_strata.json", strata)\n    intervals, distributions, deltas = {}, {}, {}\n    r0_scenes = [scene for scene in feature_arrays if scene.endswith("_r0")]\n    for col, task in enumerate(("pref", "qual")):\n        observed = pairs["label_" + task].notna().to_numpy()\n        frame = pairs.loc[observed]\n        groups, ids = np.unique(frame["user" if task == "pref" else "item"], return_inverse=True)\n        counts = np.random.default_rng(42).multinomial(len(groups), np.full(len(groups), 1 / len(groups)), size=2000)\n        for scene in r0_scenes:\n            for model in spec.PROTOCOL["models"]:\n                prefix = scene + "__" + model + "__" + task\n                query = queries[prefix].set_index("query").loc[groups]\n                valid = query.positives.gt(0).to_numpy()\n                weights = counts[:, valid]\n                denominators = weights.sum(axis=1)\n                if (denominators == 0).any():\n                    raise ValueError("No positive ranking queries in bootstrap draw")\n                for metric in ("ndcg@5", "recall@5", "ndcg@10", "recall@10"):\n                    distributions[prefix + "__" + metric] = weights @ query.loc[valid, metric].to_numpy() / denominators\n                distributions[prefix + "__average_precision"] = previous.bootstrap_ap(\n                    frame["label_" + task].to_numpy(), predictions[scene + "__" + model][observed, col], ids, counts)\n            for metric in ("average_precision", "ndcg@5", "recall@5", "ndcg@10", "recall@10"):\n                for left, right in CONTRASTS:\n                    key = scene + "__" + left + "-" + right + "__" + task + "__" + metric\n                    diff = distributions[f"{scene}__{left}__{task}__{metric}"] - distributions[f"{scene}__{right}__{task}__{metric}"]\n                    deltas[key] = np.quantile(diff, [.025, .975]).tolist()\n            LOGGER.info("BOOTSTRAP task=%s scene=%s complete", task, scene)\n        for model in spec.PROTOCOL["models"]:\n            key = "primary_mean__" + model + "__" + task + "__ndcg@5"\n            distributions[key] = np.mean([distributions[f"{scene}_r0__{model}__{task}__ndcg@5"]\n                                           for scene in spec.PROTOCOL["primary_scenarios"]], axis=0)\n        for left, right in CONTRASTS:\n            key = "primary_mean__" + left + "-" + right + "__" + task + "__ndcg@5"\n            diff = distributions[f"primary_mean__{left}__{task}__ndcg@5"] - distributions[f"primary_mean__{right}__{task}__ndcg@5"]\n            deltas[key] = np.quantile(diff, [.025, .975]).tolist()\n    for key, values in distributions.items():\n        intervals[key] = np.quantile(values, [.025, .975]).tolist()\n    base.write_json(output / "bootstrap.json", {"seed": 42, "replicates": 2000, "intervals": intervals, "contrasts": deltas})\n    np.savez_compressed(output / "bootstrap_draws.npz", **distributions)\n    gates, sensitivity, clean_changes = {}, {}, {}\n    for model in spec.PROTOCOL["models"]:\n        gates[model] = {}\n        for task in ("pref", "qual"):\n            primary = float(np.mean([metrics[s + "_r0"][model][task]["ndcg@5"] for s in spec.PROTOCOL["primary_scenarios"]]))\n            reference = float(np.mean([metrics[s + "_r0"]["H"][task]["ndcg@5"] for s in spec.PROTOCOL["primary_scenarios"]]))\n            clean_ap = metrics["clean_r0"][model][task]["average_precision"] - metrics["clean_r0"]["H"][task]["average_precision"]\n            clean_ndcg = metrics["clean_r0"][model][task]["ndcg@5"] - metrics["clean_r0"]["H"][task]["ndcg@5"]\n            gates[model][task] = {"clean_ap_change": clean_ap, "clean_ndcg5_change": clean_ndcg,\n                                  "primary_ndcg5": primary, "primary_gain": primary - reference,\n                                  "point_estimate_gate_pass": bool(clean_ap >= -.02 and clean_ndcg >= -.02 and primary - reference >= .03)}\n            for scene in r0_scenes:\n                clean_changes[f"{scene}__{model}__{task}"] = {metric: metrics[scene][model][task][metric] - metrics["clean_r0"][model][task][metric]\n                    for metric in ("average_precision", "ndcg@5", "recall@5", "recall@10")}\n            for scenario in ("random_10", "random_30", "random_50", "mixed_optional", "job_random_30"):\n                sensitivity[f"{scenario}__{model}__{task}"] = {metric: [min(metrics[f"{scenario}_r{r}"][model][task][metric] for r in range(5)),\n                    max(metrics[f"{scenario}_r{r}"][model][task][metric] for r in range(5))]\n                    for metric in ("average_precision", "ndcg@5", "recall@5")}\n    base.write_json(output / "gates.json", gates)\n    base.write_json(output / "mask_sensitivity.json", sensitivity)\n    base.write_json(output / "change_from_clean.json", clean_changes)\n    return metrics\n'
module = types.ModuleType('jobrec_missing_eval')
exec(compile(source, 'jobrec_missing_eval.py', 'exec'), module.__dict__)
sys.modules['jobrec_missing_eval'] = module


In [ ]:
"""职责：执行已批准的v4缺失资料训练与历史基线核验。
实现：固定父产物哈希，重算资料遮蔽特征，训练8个树模型，保存预测并统一分析。
关联：jobrec_v1/v2复用历史协议；jobrec_missing定义缺失语义；jobrec_missing_eval分析输出。
目录：verify_history、prepare_scenes、train_model、main。
变量索引：LOGGER为无凭据阶段日志；其余实验参数由spec.PROTOCOL/RANKER_CONFIG提供。
约束：不改变历史模型、划分、标签、树超参数；失败保存阶段且抛出，无重试或回退。
"""

import hashlib
import importlib.metadata
import json
import logging
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

import jobrec_v1 as base
import jobrec_v2 as previous
import jobrec_missing as spec
import jobrec_missing_eval as evaluation

LOGGER = logging.getLogger("jobrec_v4")


def verify_history(folder: Path) -> None:
    """功能：核验v3历史树与结果身份；输入父目录；输出无。
    逻辑：逐文件SHA256比对冻结表；约束：任何不符即失败，不接受同名更新产物。
    """
    for name, expected in spec.HISTORICAL.items():
        if hashlib.sha256((folder / name).read_bytes()).hexdigest() != expected:
            raise ValueError(f"v3 historical artifact changed: {name}")
    LOGGER.info("Historical v3 artifacts verified")


def prepare_scenes(pairs: pd.DataFrame, users: pd.DataFrame, jobs: pd.DataFrame,
                   clean: np.ndarray, output: Path) -> dict:
    """功能：冻结所有缺失场景；输入完整数据与参考特征/输出目录；输出场景到全配对特征映射。
    逻辑：实体掩码→原始字段隐藏→特征重算，并保存实体ID、掩码和摘要；约束：clean须逐项相同。
    训练场景命名空间独立；原始档案不写输出；保存的特征含NaN，不用NaN填补标签。
    """
    data, masks, audits = {}, {}, {}
    settings = spec.scenarios() + [("training_mix", 0)]
    for scenario, repeat in settings:
        key = f"{scenario}_r{repeat}"
        um, jm, assignments = spec.make_masks(users, jobs, scenario, repeat,
                                              "training" if scenario == "training_mix" else "evaluation")
        hidden_u = spec.hide_profiles(users, um, spec.USER_BLOCKS)
        hidden_j = spec.hide_profiles(jobs, jm, spec.JOB_BLOCKS)
        matrix = spec.features(pairs, hidden_u, hidden_j)
        if scenario == "clean":
            np.testing.assert_array_equal(matrix, clean)
        data[key] = matrix
        masks[key + "__users"], masks[key + "__jobs"] = um, jm
        audits[key] = {"user_blocks_hidden": um.sum(axis=0).tolist(), "job_blocks_hidden": jm.sum(axis=0).tolist(),
                       "feature_missing_counts": np.isnan(matrix).sum(axis=0).tolist(),
                       "user_assignments": pd.Series(assignments).value_counts().to_dict()}
        if scenario == "training_mix":
            train_users = np.unique(pairs.loc[pairs.split.eq("train"), "u_idx"])
            audits[key]["labeled_training_user_assignments"] = pd.Series(np.asarray(assignments)[train_users]).value_counts().to_dict()
        LOGGER.info("FEATURES scene=%s missing=%s", key, audits[key]["feature_missing_counts"])
    np.savez_compressed(output / "masks.npz", **masks)
    users[["student_id"]].to_csv(output / "candidate_ids.csv", index=False)
    jobs[["job_id"]].to_csv(output / "job_ids.csv", index=False)
    base.write_json(output / "mask_audit.json", audits)
    np.savez_compressed(output / "training_features.npz", clean=data["clean_r0"], mixed=data["training_mix_r0"])
    return data


def train_model(model_name: str, pairs: pd.DataFrame, data: dict, folder: Path) -> list:
    """功能：训练一个预登记双向对照；输入模型名/配对/场景/模型目录；输出两个已裁剪Booster。
    逻辑：A用完整、BCD用同一混合资料；按任务查询成组；三种验证AP等权选20轮网格。
    约束：只用train拟合、validation选轮次；不复制配对；并列选更早；模型保存后重新加载。
    """
    import lightgbm as lgb

    outputs, history = [], {}
    train_features = spec.feature_view(data["clean_r0" if model_name == "A" else "training_mix_r0"], model_name)
    for task in ("pref", "qual"):
        label, group = "label_" + task, "user" if task == "pref" else "item"
        train = pairs.loc[pairs.split.eq("train") & pairs[label].notna()].sort_values(group, kind="stable")
        validation = pairs.loc[pairs.split.eq("validation") & pairs[label].notna()]
        sizes = train.groupby(group, sort=False).size().to_numpy()
        if sizes.sum() != len(train):
            raise ValueError("Invalid query grouping")
        model = lgb.LGBMRanker(**spec.RANKER_CONFIG)
        model.fit(train_features[train.index], train[label].astype(int), group=sizes)
        records = []
        for iteration in spec.PROTOCOL["tree_grid"]:
            aps = {}
            for scenario in spec.PROTOCOL["validation_scenarios"]:
                matrix = spec.feature_view(data[scenario + "_r0"][validation.index], model_name)
                scores = model.booster_.predict(matrix, num_iteration=iteration, num_threads=2)
                aps[scenario] = float(average_precision_score(validation[label], scores))
            records.append({"iteration": iteration, "scenario_ap": aps, "mean_ap": float(np.mean(list(aps.values())))})
        selected = max(records, key=lambda row: row["mean_ap"])
        target = folder / (task + ".txt")
        model.booster_.save_model(str(target), num_iteration=selected["iteration"])
        booster = lgb.Booster(model_file=str(target))
        outputs.append(booster)
        history[task] = {"selection": selected, "history": records, "training_pairs": len(train),
                         "training_queries": len(sizes), "actual_params": model.get_params(),
                         "features": train_features.shape[1]}
        LOGGER.info("TRAIN model=%s task=%s selected=%s", model_name, task, selected)
    base.write_json(folder / "history.json", history)
    return outputs


def main() -> None:
    """功能：运行完整v4；输入Kaggle父kernel、网络和已批准协议；输出jobrec_v4产物。
    逻辑：核验→环境→数据→clean复现→缺失诊断→8次拟合→评测；每阶段日志及失败状态落盘。
    约束：沿用历史环境引导；树模型仅CPU两线程，不声称使用GPU训练；不执行文本或面试模型。
    """
    output = Path("/kaggle/working/jobrec_v4")
    output.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(level=logging.INFO, force=True, format="%(asctime)s %(levelname)s %(message)s",
                        handlers=[logging.StreamHandler(sys.stdout), logging.FileHandler(output / "execution.log")])
    started, stage = time.monotonic(), "protocol"
    base.write_json(output / "protocol.json", spec.PROTOCOL)
    base.write_json(output / "protocol_hash.json", {"sha256": hashlib.sha256((output / "protocol.json").read_bytes()).hexdigest()})
    try:
        roots = list(Path("/kaggle/input").rglob("jobrec_v1/config.json"))
        histories = list(Path("/kaggle/input").rglob("jobrec_v3/lambdarank/pref.txt"))
        if len(roots) != 1 or len(histories) != 1:
            raise ValueError(f"Expected one v1/v3 parent; found {len(roots)}/{len(histories)}")
        parent, history_parent = roots[0].parent, histories[0].parent.parent
        previous.verify_inputs(parent)
        verify_history(history_parent)
        stage = "environment"
        base.bootstrap(output)
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "lightgbm==4.6.0"], check=True)
        import lightgbm as lgb

        base.write_json(output / "ranker_environment.json", {"lightgbm": importlib.metadata.version("lightgbm"),
                        "compute": "CPU, n_jobs=2; no embedding or neural training"})
        stage = "data"
        frames = base.download_data(output)
        if json.loads((output / "data_manifest.json").read_text()) != json.loads((parent / "data_manifest.json").read_text()):
            raise ValueError("Source data changed")
        pairs, users, jobs, _, clean = base.prepare_data(frames, output)
        pd.testing.assert_frame_equal(pairs, pd.read_csv(parent / "pair_splits.csv"), check_dtype=False)
        np.testing.assert_array_equal(spec.features(pairs, users, jobs), clean)
        test = pairs.loc[pairs.split.eq("test")].copy()
        test_ids = test.index.to_numpy()
        test.to_csv(output / "test_pairs.csv", index=False)
        models, predictions = {}, {}
        historical_folder = output / "H"
        historical_folder.mkdir()
        models["H"] = []
        for task in ("pref", "qual"):
            shutil.copy2(history_parent / "lambdarank" / (task + ".txt"), historical_folder / (task + ".txt"))
            models["H"].append(lgb.Booster(model_file=str(historical_folder / (task + ".txt"))))
        clean_scores = np.column_stack([model.predict(clean[test_ids], num_threads=2) for model in models["H"]])
        old = pd.read_csv(history_parent / "test_predictions.csv")
        pd.testing.assert_frame_equal(test[["user", "item"]].reset_index(drop=True), old[["user", "item"]])
        np.testing.assert_allclose(clean_scores, old[["lambdarank_taskwise_pref", "lambdarank_taskwise_qual"]].to_numpy(), rtol=1e-12, atol=1e-12)
        reference_metrics = json.loads((history_parent / "metrics.json").read_text())["lambdarank_taskwise"]
        actual = base.evaluate_scores(test, clean_scores, "test")
        for task in actual:
            for metric, value in actual[task].items():
                np.testing.assert_allclose(value, reference_metrics[task][metric], rtol=0, atol=1e-12)
        base.write_json(output / "baseline_verification.json", {"features_exact": True, "predictions_tolerance": 1e-12,
                        "metrics_tolerance": 1e-12, "metrics": actual, "history_hashes": spec.HISTORICAL})
        LOGGER.info("BASELINE clean features, predictions and metrics verified")
        stage = "mask_features"
        data = prepare_scenes(pairs, users, jobs, clean, output)
        scenes = {key: matrix[test_ids] for key, matrix in data.items() if key != "training_mix_r0"}
        np.savez_compressed(output / "test_features.npz", **scenes)
        stage = "historical_diagnostics"
        diagnostics = {}
        for scene, matrix in scenes.items():
            values = np.column_stack([model.predict(matrix, num_threads=2) for model in models["H"]])
            predictions[scene + "__H"] = values
            diagnostics[scene] = base.evaluate_scores(test, values, "test")
        base.write_json(output / "historical_diagnostics.json", diagnostics)
        LOGGER.info("HISTORICAL missing diagnostics complete; training protocol unchanged")
        for name in "ABCD":
            stage = "train_" + name
            folder = output / name
            folder.mkdir()
            models[name] = train_model(name, pairs, data, folder)
            for scene, matrix in scenes.items():
                inputs = spec.feature_view(matrix, name)
                predictions[scene + "__" + name] = np.column_stack([model.predict(inputs, num_threads=2) for model in models[name]])
        np.savez_compressed(output / "predictions.npz", **predictions)
        stage = "evaluation"
        evaluation.analyze(test.reset_index(drop=True), predictions, scenes, output)
        base.write_json(output / "status.json", {"status": "complete", "elapsed_seconds": time.monotonic() - started,
                        "trained_models": 8, "scenarios_including_mask_repeats": len(scenes)})
        LOGGER.info("JOBREC_V4_COMPLETE")
    except Exception as error:
        base.write_json(output / "status.json", {"status": "failed", "stage": stage, "error": str(error)})
        LOGGER.exception("v4 failed at %s; no fallback/retry", stage)
        raise


if __name__ == "__main__":
    main()
